# 02 · Silver — limpo, tipado, deduplicado

Tudo o que a Silver faz tem uma **regra explícita**. O que viola a regra não some: vai para a **quarentena**.

| Tabela | Regras |
|---|---|
| clientes | `cliente_id` int; `estado` em maiúsculas; e-mail em minúsculas, inválido vira `NULL`; datas em 3 formatos; fica a versão mais recente por `cliente_id` |
| produtos | `preco` e `custo` > 0, senão quarentena |
| pedidos | tipos corretos; deduplica por `pedido_id`; **cliente inexistente → quarentena**; `itens` explodem em `itens_pedido` |
| avaliacoes | `nota` entre 1 e 5, senão quarentena; deduplica por `avaliacao_id` |

A Silver é reconstruída a partir de **toda** a Bronze (todos os lotes), então é segura para reexecutar.

In [2]:
from common import lake_local as L

print("Endpoint:", L.MINIO_ENDPOINT)

try:
    resposta = L._s3().list_buckets()
    buckets = [b["Name"] for b in resposta["Buckets"]]
    print("Buckets encontrados:", buckets)
except Exception as e:
    print("Erro ao acessar o MinIO:")
    print(type(e).__name__, str(e))

Endpoint: http://localhost:9003
Erro ao acessar o MinIO:
ClientError An error occurred (InvalidArgument) when calling the ListBuckets operation: S3 API Requests must be made to API port.


In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from common import lake_local as L, medalhao as M
from pyspark.sql import functions as F

spark = L.get_spark()

print(
    "Spark", spark.version,
    "| master:", spark.sparkContext.master,
    "| storage:", L.STORAGE
)

Spark 3.5.3 | master: local[2] | storage: minio


In [2]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))          # para importar a pasta common/
from common import lake_local as L, medalhao as M
from pyspark.sql import functions as F
spark = L.get_spark()                              # PySpark local: sem cluster
print("Spark", spark.version, "| master:", spark.sparkContext.master, "| storage:", L.STORAGE)

Spark 3.5.3 | master: local[2] | storage: minio


## Antes da função: faça na mão
Na aula de Spark você explodiu a coluna `itens`. Agora vamos fazer isso **a partir da Bronze**, que tem uma diferença: ela guarda os pedidos duplicados.
Se explodirmos antes de deduplicar, cada item duplicado conta duas vezes e o faturamento fica inflado.

In [4]:
print("Tabelas no namespace Bronze:")

spark.sql("SHOW TABLES IN lake.bronze").show(truncate=False)

Tabelas no namespace Bronze:
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [5]:
print("Arquivos no warehouse:")

for chave, tamanho in L.list_warehouse():
    print(f"{tamanho:>10,} bytes  {chave}")

Arquivos no warehouse:
         0 bytes  lakehouse/bronze/
         0 bytes  lakehouse/gold/
         0 bytes  lakehouse/silver/


In [6]:
print("Arquivos do lote 1 no landing:")

for chave, tamanho in L.list_landing("mercadonordeste/lote1"):
    print(f"{tamanho:>10,} bytes  {chave}")

Arquivos do lote 1 no landing:


In [7]:
import inspect
from common import medalhao as M

print(inspect.getsource(M.ingest_lote))

def ingest_lote(spark, batch, landing_uri):
    """Ingere os 4 arquivos de um lote da landing para a Bronze."""
    base = landing_uri(f"mercadonordeste/{batch}")
    res = {}
    res["clientes"] = to_bronze(spark, spark.read.option("header", True).csv(f"{base}/clientes.csv"),
                                "lake.bronze.clientes", batch)
    if batch == "lote1":
        res["produtos"] = to_bronze(spark, spark.read.option("header", True).csv(f"{base}/produtos.csv"),
                                    "lake.bronze.produtos", batch)
    res["pedidos"] = to_bronze(spark, spark.read.json(f"{base}/pedidos.jsonl"), "lake.bronze.pedidos", batch)
    res["avaliacoes"] = to_bronze(spark, spark.read.option("header", True).csv(f"{base}/avaliacoes.csv"),
                                  "lake.bronze.avaliacoes", batch)
    return res



In [8]:
L.MINIO_ENDPOINT = "http://localhost:9002"

print("Endpoint:", L.MINIO_ENDPOINT)
print("Arquivos locais:", list(pasta.glob("*")))

Endpoint: http://localhost:9002


NameError: name 'pasta' is not defined

In [9]:
from pathlib import Path
from common import lake_local as L

# Corrige o endpoint da API S3 do MinIO
L.MINIO_ENDPOINT = "http://localhost:9002"

# Define a pasta local do lote 1
pasta = L.DATA_DIR / "lote1"

print("Endpoint:", L.MINIO_ENDPOINT)
print("Pasta:", pasta)
print("Pasta existe:", pasta.is_dir())

if pasta.is_dir():
    for arquivo in pasta.iterdir():
        print(
            arquivo.name,
            f"- {arquivo.stat().st_size:,} bytes"
            if arquivo.is_file() else "(pasta)"
        )

Endpoint: http://localhost:9002
Pasta: E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao\data\lote1
Pasta existe: True
avaliacoes.csv - 529,223 bytes
clientes.csv - 420,891 bytes
pedidos.jsonl - 7,571,082 bytes
produtos.csv - 9,044 bytes


In [11]:
for chave, tamanho in L.list_landing("mercadonordeste/lote1"):
    print(f"{tamanho:>10,} bytes  {chave}")

   529,223 bytes  mercadonordeste/lote1/avaliacoes.csv
   420,891 bytes  mercadonordeste/lote1/clientes.csv
 7,571,082 bytes  mercadonordeste/lote1/pedidos.jsonl
     9,044 bytes  mercadonordeste/lote1/produtos.csv


In [10]:
n = L.upload_to_landing(pasta, "mercadonordeste/lote1")
print(n, "arquivos enviados")

4 arquivos enviados


In [14]:
from common import lake_local as L

L.MINIO_ENDPOINT = "http://localhost:9002"

arquivos = L.list_landing("mercadonordeste/lote1")

print("Total de arquivos na Landing:", len(arquivos))

for chave, tamanho in arquivos:
    print(chave, tamanho, "bytes")

Total de arquivos na Landing: 4
mercadonordeste/lote1/avaliacoes.csv 529223 bytes
mercadonordeste/lote1/clientes.csv 420891 bytes
mercadonordeste/lote1/pedidos.jsonl 7571082 bytes
mercadonordeste/lote1/produtos.csv 9044 bytes


In [15]:
from common import medalhao as M

contagens = M.ingest_lote(
    spark,
    "lote1",
    L.landing_uri
)

print(contagens)

{'clientes': 5000, 'produtos': 200, 'pedidos': 30150, 'avaliacoes': 6489}


In [16]:
spark.sql("SHOW TABLES IN lake.bronze").show(truncate=False)

+---------+----------+-----------+
|namespace|tableName |isTemporary|
+---------+----------+-----------+
|bronze   |clientes  |false      |
|bronze   |avaliacoes|false      |
|bronze   |produtos  |false      |
|bronze   |pedidos   |false      |
+---------+----------+-----------+



In [17]:
from common import lake_local as L, medalhao as M

# Garante o endpoint correto da API S3
L.MINIO_ENDPOINT = "http://localhost:9002"

# Carrega os dados do lote 1 para a Bronze
contagens = M.ingest_lote(
    spark,
    "lote1",
    L.landing_uri
)

print(contagens)

{'clientes': 5000, 'produtos': 200, 'pedidos': 30150, 'avaliacoes': 6489}


In [18]:
spark.sql("SHOW TABLES IN lake.bronze").show(truncate=False)

+---------+----------+-----------+
|namespace|tableName |isTemporary|
+---------+----------+-----------+
|bronze   |clientes  |false      |
|bronze   |avaliacoes|false      |
|bronze   |produtos  |false      |
|bronze   |pedidos   |false      |
+---------+----------+-----------+



In [19]:
brz = spark.table("lake.bronze.pedidos")

print("Total de linhas na Bronze:", brz.count())
brz.show(5, truncate=False)

Total de linhas na Bronze: 30150
+----------+-------------------+---------------+--------------------------------------------------+---------+---------+-----------+--------------------------+-------------------------------------------------+---------+
|cliente_id|data_pedido        |forma_pagamento|itens                                             |pedido_id|status   |valor_frete|_ingest_ts                |_source_file                                     |_batch_id|
+----------+-------------------+---------------+--------------------------------------------------+---------+---------+-----------+--------------------------+-------------------------------------------------+---------+
|2769      |2025-11-28T17:05:06|pix            |[{211.62, 6, 2}]                                  |123912   |devolvido|24.9       |2026-10-24 13:27:10.620971|s3a://landing/mercadonordeste/lote1/pedidos.jsonl|lote1    |
|3227      |2025-12-22T08:51:13|cartao_debito  |[{824.72, 39, 1}]                          

In [20]:
from pyspark.sql.window import Window

brz = spark.table("lake.bronze.pedidos")
print("Bronze:", brz.count(), "linhas |", brz.select("pedido_id").distinct().count(), "pedidos distintos")

# 1) deduplica por pedido_id, ficando com a versão mais recente
w = Window.partitionBy("pedido_id").orderBy(F.col("_ingest_ts").desc())
pedidos_unicos = brz.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
print("Duplicados descartados:", brz.count() - pedidos_unicos.count())

# 2) explode itens (posexplode guarda a posição do item no pedido)
itens = (pedidos_unicos
    .select("pedido_id", "cliente_id", "data_pedido", "status",
            F.posexplode("itens").alias("item_seq", "item"))
    .select("pedido_id", "cliente_id", "data_pedido", "status", "item_seq",
            F.col("item.produto_id").alias("produto_id"),
            F.col("item.qtd").alias("qtd"),
            F.col("item.preco_unit").alias("preco_unit"))
    .withColumn("valor_item", F.round(F.col("qtd") * F.col("preco_unit"), 2)))

print("pedidos:", pedidos_unicos.count(), "| itens:", itens.count())
itens.show(5, truncate=False)

Bronze: 30150 linhas | 30000 pedidos distintos
Duplicados descartados: 150
pedidos: 30000 | itens: 50105
+---------+----------+-------------------+---------+--------+----------+---+----------+----------+
|pedido_id|cliente_id|data_pedido        |status   |item_seq|produto_id|qtd|preco_unit|valor_item|
+---------+----------+-------------------+---------+--------+----------+---+----------+----------+
|100001   |1257      |2025-09-21T23:11:30|entregue |0       |200       |1  |595.8     |595.8     |
|100002   |1959      |2025-06-05T23:59:12|cancelado|0       |101       |1  |872.69    |872.69    |
|100002   |1959      |2025-06-05T23:59:12|cancelado|1       |82        |3  |220.7     |662.1     |
|100002   |1959      |2025-06-05T23:59:12|cancelado|2       |57        |1  |460.02    |460.02    |
|100002   |1959      |2025-06-05T23:59:12|cancelado|3       |142       |2  |577.76    |1155.52   |
+---------+----------+-------------------+---------+--------+----------+---+----------+----------+
only

### Checagens de qualidade nos itens
Mesmas três perguntas da aula (valores inválidos, chave duplicada, produto inexistente), mas agora quem falha **não some**: ganha um motivo e vai para a quarentena.

In [21]:
produtos = (spark.table("lake.bronze.produtos")
            .select(F.col("produto_id").cast("long").alias("produto_id"))
            .distinct().withColumn("_existe", F.lit(True)))

valor_ok = (F.col("qtd") > 0) & (F.col("preco_unit") > 0) & F.col("produto_id").isNotNull()

itens_f = (itens.join(produtos, "produto_id", "left")
    .withColumn("motivo",
        F.when(F.coalesce(~valor_ok, F.lit(True)), "valor_invalido")
         .when(F.col("_existe").isNull(), "produto_inexistente")))

itens_ok  = itens_f.filter("motivo IS NULL").drop("motivo", "_existe")
itens_rej = itens_f.filter("motivo IS NOT NULL")

print("itens válidos:", itens_ok.count(), "| rejeitados:", itens_rej.count())
itens_rej.groupBy("motivo").count().show()

# chave composta única (pedido_id, item_seq) nos válidos
dup = itens_ok.groupBy("pedido_id", "item_seq").count().filter("count > 1").count()
print("chaves duplicadas:", dup, "->", "OK" if dup == 0 else "ERRO")

itens válidos: 50105 | rejeitados: 0
+------+-----+
|motivo|count|
+------+-----+
+------+-----+

chaves duplicadas: 0 -> OK


In [22]:
# Salvar na Silver

itens_ok.writeTo("lake.silver.itens_pedido_aula").using("iceberg").createOrReplace()
spark.table("lake.silver.itens_pedido_aula").count()

50105

In [23]:
resumo = M.build_silver(spark)
resumo

{'clientes': 5000,
 'produtos': 197,
 'pedidos': 29960,
 'itens_pedido': 50030,
 'avaliacoes': 6464,
 'quarentena': 68}

## Reconciliação: tudo que entrou tem que estar em algum lugar
Regra de ouro: **entrada = saída + quarentena**. Se a conta não fecha, o pipeline está perdendo dado sem avisar.

In [24]:
n_bronze = spark.table("lake.bronze.pedidos").select("pedido_id").distinct().count()
n_silver = spark.table("lake.silver.pedidos").count()
n_quar   = spark.table("lake.silver.quarentena").filter("origem = 'pedidos_cliente_orfao'").count()
print(f"pedidos : {n_bronze} = {n_silver} (silver) + {n_quar} (quarentena) -> {n_bronze == n_silver + n_quar}")

n_mao    = spark.table("lake.silver.itens_pedido_aula").count()
n_funcao = spark.table("lake.silver.itens_pedido").count()
print(f"itens   : feito na mão = {n_mao} | feito pela função = {n_funcao}")

pedidos : 30000 = 29960 (silver) + 40 (quarentena) -> True
itens   : feito na mão = 50105 | feito pela função = 50030


## O que foi para a quarentena?

In [25]:
spark.table("lake.silver.quarentena").groupBy("origem").count().show(truncate=False)
spark.table("lake.silver.quarentena").show(5, truncate=False)

+------------------------+-----+
|origem                  |count|
+------------------------+-----+
|pedidos_cliente_orfao   |40   |
|produtos                |3    |
|avaliacoes_nota_invalida|25   |
+------------------------+-----+

+---------------------+-------------------------------------------------+--------------------------+
|origem               |registro                                         |_quarentena_ts            |
+---------------------+-------------------------------------------------+--------------------------+
|produtos             |{"produto_id":55,"preco":-418.73,"custo":328.22} |2026-10-24 13:30:04.268907|
|produtos             |{"produto_id":114,"preco":-462.01,"custo":345.79}|2026-10-24 13:30:04.268907|
|produtos             |{"produto_id":173,"preco":-308.2,"custo":231.93} |2026-10-24 13:30:04.268907|
|pedidos_cliente_orfao|{"pedido_id":103054,"cliente_id":999999}         |2026-10-24 13:30:04.268907|
|pedidos_cliente_orfao|{"pedido_id":104139,"cliente_id":99999

## Data Quality: medir, não só limpar
`DQ` aplica regras e devolve um relatório. Guardar o relatório numa tabela é o começo da **observabilidade**.

In [28]:
import os
import sys

print("Python do Jupyter:", sys.executable)
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("Python configurado no Spark:",
      spark.sparkContext.getConf().get("spark.pyspark.python", "(padrão)"))

spark.sparkContext.parallelize([1, 2, 3], 1).map(
    lambda x: x + 1
).collect()

Python do Jupyter: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
PYSPARK_PYTHON: None
Python configurado no Spark: (padrão)


Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 246.0 failed 1 times, most recent failure: Lost task 0.0 in stage 246.0 (TID 161) (DESKTOP-8IQO9ET executor driver): java.net.SocketException: Connection reset
	at java.base/sun.nio.ch.NioSocketImpl.implRead(NioSocketImpl.java:328)
	at java.base/sun.nio.ch.NioSocketImpl.read(NioSocketImpl.java:355)
	at java.base/sun.nio.ch.NioSocketImpl$1.read(NioSocketImpl.java:808)
	at java.base/java.net.Socket$SocketInputStream.read(Socket.java:966)
	at java.base/java.io.BufferedInputStream.fill(BufferedInputStream.java:244)
	at java.base/java.io.BufferedInputStream.read(BufferedInputStream.java:263)
	at java.base/java.io.DataInputStream.readInt(DataInputStream.java:381)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:774)
	at org.apache.spark.api.python.PythonRunner$$anon$3.r

In [29]:
import sys
import os
import pyspark

print("Python do Jupyter:", sys.executable)
print("Versão Python:", sys.version)
print("PySpark:", pyspark.__version__)
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("PYSPARK_DRIVER_PYTHON:", os.environ.get("PYSPARK_DRIVER_PYTHON"))
print("Python do Spark:",
      spark.sparkContext.getConf().get("spark.pyspark.python", "(padrão)"))

Python do Jupyter: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
Versão Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
PySpark: 3.5.3
PYSPARK_PYTHON: None
PYSPARK_DRIVER_PYTHON: None
Python do Spark: (padrão)


In [30]:
import os
import sys

python_exe = sys.executable

os.environ["PYSPARK_PYTHON"] = python_exe
os.environ["PYSPARK_DRIVER_PYTHON"] = python_exe

print("Python configurado:", python_exe)

Python configurado: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe


In [2]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

sys.path.insert(0, os.path.abspath(".."))

from common import lake_local as L, medalhao as M
from pyspark.sql import functions as F

spark = L.get_spark()

print("Spark:", spark.version)
print("Python:", sys.executable)
print("Master:", spark.sparkContext.master)

Spark: 3.5.3
Python: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
Master: local[2]


In [3]:
spark.sparkContext.parallelize([1, 2, 3], 1).map(
    lambda x: x + 1
).collect()

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, most recent failure: Lost task 0.0 in stage 0.0 (TID 0) (DESKTOP-8IQO9ET executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:612)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:594)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:789)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(I

In [4]:
try:
    spark.sparkContext.parallelize([1, 2, 3], 1).map(
        lambda x: x + 1
    ).collect()
except Exception as e:
    print("Erro Python:", repr(e))

    java_error = getattr(e, "java_exception", None)
    nivel = 0

    while java_error is not None and nivel < 10:
        print(f"\nCausa {nivel}: {java_error.toString()}")
        java_error = java_error.getCause()
        nivel += 1

Erro Python: Py4JJavaError('An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.\n', JavaObject id=o96)

Causa 0: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 1.0 failed 1 times, most recent failure: Lost task 0.0 in stage 1.0 (TID 1) (DESKTOP-8IQO9ET executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:612)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:594)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:789)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.

In [5]:
import sys
import subprocess
from pathlib import Path

raiz = Path(r"E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao")
python_venv = raiz / ".venv" / "Scripts" / "python.exe"

print("Python do Jupyter:", sys.executable)
print("Python do projeto existe:", python_venv.exists())

for nome, exe in [
    ("Jupyter", Path(sys.executable)),
    ("Ambiente virtual", python_venv),
]:
    if exe.exists():
        resultado = subprocess.run(
            [
                str(exe), "-c",
                "import sys, pyspark; "
                "print('Python:', sys.version); "
                "print('PySpark:', pyspark.__version__); "
                "import pyspark.worker; "
                "print('Worker importado')"
            ],
            capture_output=True,
            text=True,
            timeout=30
        )

        print(f"\n--- {nome} ---")
        print("Código:", resultado.returncode)
        print(resultado.stdout)
        if resultado.stderr:
            print("Erro:", resultado.stderr)

Python do Jupyter: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
Python do projeto existe: True

--- Jupyter ---
Código: 0
Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
PySpark: 3.5.3
Worker importado


--- Ambiente virtual ---
Código: 0
Python: 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
PySpark: 3.5.3
Worker importado



In [6]:
import subprocess

resultado = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)

print(resultado.stderr or resultado.stdout)

openjdk version "17.0.20.1" 2026-08-18
OpenJDK Runtime Environment Temurin-17.0.20.1+1 (build 17.0.20.1+1)
OpenJDK 64-Bit Server VM Temurin-17.0.20.1+1 (build 17.0.20.1+1, mixed mode, sharing)



In [7]:
import os
import sys

print("Python atual:", sys.executable)
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print(
    "Python configurado no Spark:",
    spark.sparkContext.getConf().get("spark.pyspark.python", "(padrão)")
)
print("Versão do Spark:", spark.version)

Python atual: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
PYSPARK_PYTHON: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
Python configurado no Spark: (padrão)
Versão do Spark: 3.5.3


In [8]:
spark.stop()
print("Sessão Spark encerrada.")

Sessão Spark encerrada.


In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

print("Python:", sys.executable)
print("SPARK_LOCAL_IP:", os.environ["SPARK_LOCAL_IP"])

Python: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
SPARK_LOCAL_IP: 127.0.0.1


In [2]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from common import lake_local as L, medalhao as M
from pyspark.sql import functions as F

spark = L.get_spark()

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

Spark: 3.5.3
Master: local[2]


In [4]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

print("Python:", sys.executable)
print("SPARK_LOCAL_IP:", os.environ["SPARK_LOCAL_IP"])

Python: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
SPARK_LOCAL_IP: 127.0.0.1


In [2]:
from pathlib import Path

spark_temp = Path(
    r"E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao\spark_tmp"
)

spark_temp.mkdir(parents=True, exist_ok=True)

print("Diretório temporário:", spark_temp)
print("Existe:", spark_temp.exists())

Diretório temporário: E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao\spark_tmp
Existe: True


In [4]:
from pathlib import Path
import sys
import os

ROOT = Path(
    r"E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao"
)

os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Diretório atual:", os.getcwd())
print("Pasta common existe:", (ROOT / "common").is_dir())
print("Arquivo dq.py existe:", (ROOT / "common" / "dq.py").is_file())
print("Arquivos da pasta common:")

if (ROOT / "common").is_dir():
    print([p.name for p in (ROOT / "common").iterdir()])

Diretório atual: E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao
Pasta common existe: True
Arquivo dq.py existe: True
Arquivos da pasta common:
['dq.py', 'lake_local.py', 'medalhao.py', '__init__.py', '__pycache__']


In [5]:
from common.dq import DQ

print("Importação da classe DQ realizada com sucesso!")

Importação da classe DQ realizada com sucesso!


In [7]:
from common import lake_local as L

spark = L.get_spark()

print("Spark iniciado:", spark.version)
print("Modo de execução:", spark.sparkContext.master)

Spark iniciado: 3.5.3
Modo de execução: local[2]


In [8]:
spark.sql("SHOW TABLES IN lake.silver").show(truncate=False)

+---------+-----------------+-----------+
|namespace|tableName        |isTemporary|
+---------+-----------------+-----------+
|silver   |avaliacoes       |false      |
|silver   |produtos         |false      |
|silver   |itens_pedido_aula|false      |
|silver   |clientes         |false      |
|silver   |pedidos          |false      |
|silver   |quarentena       |false      |
|silver   |itens_pedido     |false      |
+---------+-----------------+-----------+



In [9]:
from common.dq import DQ

cli = spark.table("lake.silver.clientes")
ped = spark.table("lake.silver.pedidos")
av = spark.table("lake.silver.avaliacoes")

r = (
    DQ(spark, cli, "silver.clientes")
    .not_null("cliente_id")
    .unique("cliente_id")
    .not_null("email", "aviso")
)

r2 = (
    DQ(spark, ped, "silver.pedidos")
    .not_null("pedido_id")
    .unique("pedido_id")
    .in_set(
        "status",
        ["pago", "enviado", "entregue", "cancelado", "devolvido"]
    )
)

r3 = (
    DQ(spark, av, "silver.avaliacoes")
    .unique("avaliacao_id")
    .between("nota", 1, 5)
)

rel = r.report()

for resultado in (r2, r3):
    rel = rel.unionByName(resultado.report())

rel.show(truncate=False)

rel.writeTo("lake.silver.dq_resultados") \
   .using("iceberg") \
   .createOrReplace()

Py4JJavaError: An error occurred while calling o187.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 39.0 failed 1 times, most recent failure: Lost task 0.0 in stage 39.0 (TID 22) (DESKTOP-8IQO9ET executor driver): java.net.SocketException: Connection reset
	at java.base/sun.nio.ch.NioSocketImpl.implRead(NioSocketImpl.java:328)
	at java.base/sun.nio.ch.NioSocketImpl.read(NioSocketImpl.java:355)
	at java.base/sun.nio.ch.NioSocketImpl$1.read(NioSocketImpl.java:808)
	at java.base/java.net.Socket$SocketInputStream.read(Socket.java:966)
	at java.base/java.io.BufferedInputStream.fill(BufferedInputStream.java:244)
	at java.base/java.io.BufferedInputStream.read(BufferedInputStream.java:263)
	at java.base/java.io.DataInputStream.readInt(DataInputStream.java:381)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:774)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache

In [10]:


from py4j.protocol import Py4JJavaError

try:
    rel.show(truncate=False)

except Py4JJavaError as e:
    print("MENSAGEM PRINCIPAL:")
    print(e.java_exception.toString())

    print("\nCAUSA MAIS PROFUNDA:")
    causa = e.java_exception

    while causa.getCause() is not None:
        causa = causa.getCause()

    print(causa.toString())

    print("\nFINAL DO TRACEBACK:")
    print("\n".join(str(e).splitlines()[-40:]))


MENSAGEM PRINCIPAL:
org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 40.0 failed 1 times, most recent failure: Lost task 0.0 in stage 40.0 (TID 23) (DESKTOP-8IQO9ET executor driver): java.net.SocketException: Connection reset
	at java.base/sun.nio.ch.NioSocketImpl.implRead(NioSocketImpl.java:328)
	at java.base/sun.nio.ch.NioSocketImpl.read(NioSocketImpl.java:355)
	at java.base/sun.nio.ch.NioSocketImpl$1.read(NioSocketImpl.java:808)
	at java.base/java.net.Socket$SocketInputStream.read(Socket.java:966)
	at java.base/java.io.BufferedInputStream.fill(BufferedInputStream.java:244)
	at java.base/java.io.BufferedInputStream.read(BufferedInputStream.java:263)
	at java.base/java.io.DataInputStream.readInt(DataInputStream.java:381)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:774)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNex

In [11]:

import sys
import os
import pyspark

print("Python do Jupyter:", sys.executable)
print("Versão do Python:", sys.version)
print("Versão do PySpark:", pyspark.__version__)
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("Python configurado no Spark:",
      spark.sparkContext.getConf().get("spark.pyspark.python", "(padrão)"))


Python do Jupyter: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
Versão do Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Versão do PySpark: 3.5.3
PYSPARK_PYTHON: None
Python configurado no Spark: (padrão)


In [12]:

resultado = (
    spark.sparkContext
    .parallelize([1, 2, 3], 1)
    .map(lambda x: x + 1)
    .collect()
)

print(resultado)


Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 41.0 failed 1 times, most recent failure: Lost task 0.0 in stage 41.0 (TID 24) (DESKTOP-8IQO9ET executor driver): java.net.SocketException: Connection reset
	at java.base/sun.nio.ch.NioSocketImpl.implRead(NioSocketImpl.java:328)
	at java.base/sun.nio.ch.NioSocketImpl.read(NioSocketImpl.java:355)
	at java.base/sun.nio.ch.NioSocketImpl$1.read(NioSocketImpl.java:808)
	at java.base/java.net.Socket$SocketInputStream.read(Socket.java:966)
	at java.base/java.io.BufferedInputStream.fill(BufferedInputStream.java:244)
	at java.base/java.io.BufferedInputStream.read(BufferedInputStream.java:263)
	at java.base/java.io.DataInputStream.readInt(DataInputStream.java:381)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:774)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read

In [1]:

from common import lake_local as L
import sys

spark = L.get_spark()

print("Python:", sys.executable)
print("Python dos workers:",
      spark.sparkContext.getConf().get("spark.pyspark.python"))

resultado = (
    spark.sparkContext
    .parallelize([1, 2, 3], 1)
    .map(lambda x: x + 1)
    .collect()
)

print("Resultado:", resultado)


ModuleNotFoundError: No module named 'common'

In [2]:

import os
import sys
from pathlib import Path

ROOT = Path(
    r"E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao"
)

print("Projeto existe:", ROOT.exists())
print("Pasta common existe:", (ROOT / "common").is_dir())
print("Arquivo lake_local.py existe:", (ROOT / "common" / "lake_local.py").is_file())

os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Diretório atual:", os.getcwd())
print("Caminho adicionado ao Python:", str(ROOT))


Projeto existe: True
Pasta common existe: True
Arquivo lake_local.py existe: True
Diretório atual: E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao
Caminho adicionado ao Python: E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao


In [3]:

from common import lake_local as L

print("Importação realizada com sucesso!")
print("Endpoint do MinIO:", L.MINIO_ENDPOINT)


Importação realizada com sucesso!
Endpoint do MinIO: http://localhost:9002


In [4]:
spark = L.get_spark()

print("Versão do Spark:", spark.version)
print("Modo:", spark.sparkContext.master)
print(
    "Python dos workers:",
    spark.sparkContext.getConf().get("spark.pyspark.python")
)

Versão do Spark: 3.5.3
Modo: local[2]
Python dos workers: None


In [5]:
resultado = (
    spark.sparkContext
    .parallelize([1, 2, 3], 1)
    .map(lambda x: x + 1)
    .collect()
)

print("Resultado:", resultado)

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, most recent failure: Lost task 0.0 in stage 0.0 (TID 0) (DESKTOP-8IQO9ET executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:612)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:594)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:789)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(I

In [6]:

import os
import sys
import subprocess
import inspect

from common import lake_local as L

print("=== ARQUIVO CARREGADO ===")
print(L.__file__)

print("\n=== CONFIGURAÇÃO DO PYTHON ===")
print("Executável:", sys.executable)
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("Versão Python do Spark:", spark.sparkContext.pythonVer)

print("\n=== CONFIGURAÇÕES PYSPARK ===")
for chave, valor in spark.sparkContext.getConf().getAll():
    if "python" in chave.lower():
        print(chave, "=", valor)

print("\n=== TESTE DO PYTHON ===")
teste_python = subprocess.run(
    [
        sys.executable,
        "-c",
        "import sys, pyspark; print(sys.executable); print(pyspark.__version__)"
    ],
    capture_output=True,
    text=True,
    timeout=20
)

print("Código de saída:", teste_python.returncode)
print("Saída:", teste_python.stdout)
print("Erro:", teste_python.stderr)

print("\n=== TESTE SQL DO SPARK ===")
spark.range(5).show()


=== ARQUIVO CARREGADO ===
E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao\common\lake_local.py

=== CONFIGURAÇÃO DO PYTHON ===
Executável: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
PYSPARK_PYTHON: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
Versão Python do Spark: 3.12

=== CONFIGURAÇÕES PYSPARK ===

=== TESTE DO PYTHON ===
Código de saída: 0
Saída: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
3.5.3

Erro: 

=== TESTE SQL DO SPARK ===
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [1]:
import sys
from common import lake_local as L

spark = L.get_spark()

print("Python:", sys.executable)
print("Worker Python:", spark.sparkContext.pythonExec)
print("Versão Spark:", spark.version)

resultado = (
    spark.sparkContext
    .parallelize([1, 2, 3], 1)
    .map(lambda x: x + 1)
    .collect()
)

print("Resultado:", resultado)

ModuleNotFoundError: No module named 'common'

In [2]:

import os
import sys
from pathlib import Path

ROOT = Path(
    r"E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao"
)

# Confere se o projeto e os arquivos existem
assert ROOT.is_dir(), f"Pasta do projeto não encontrada: {ROOT}"
assert (ROOT / "common" / "lake_local.py").is_file(), (
    "O arquivo common/lake_local.py não foi encontrado."
)

# Define a pasta do projeto como diretório atual
os.chdir(ROOT)

# Adiciona a pasta do projeto ao caminho de importação do Python
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Diretório atual:", os.getcwd())
print("Pasta common encontrada:", (ROOT / "common").is_dir())


Diretório atual: E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao
Pasta common encontrada: True


In [3]:

from common import lake_local as L

print("Importação realizada:", L.__file__)
print("Endpoint do MinIO:", L.MINIO_ENDPOINT)


Importação realizada: E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao\common\lake_local.py
Endpoint do MinIO: http://localhost:9002


In [4]:
spark = L.get_spark()

print("Versão:", spark.version)
print("Python do worker:", spark.sparkContext.pythonExec)

Versão: 3.5.3
Python do worker: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe


In [5]:
resultado = (
    spark.sparkContext
    .parallelize([1, 2, 3], 1)
    .map(lambda x: x + 1)
    .collect()
)

print("Resultado:", resultado)

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, most recent failure: Lost task 0.0 in stage 0.0 (TID 0) (DESKTOP-8IQO9ET executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:612)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:594)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:789)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(I

In [6]:

import os
import sys
import traceback
from py4j.protocol import Py4JJavaError

print("Python:", sys.executable)
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("Python do Spark:", spark.sparkContext.pythonExec)
print("Versão do Spark:", spark.version)

# Ativa logs detalhados temporariamente
spark.sparkContext.setLogLevel("INFO")

try:
    resultado = (
        spark.sparkContext
        .parallelize([1, 2, 3], 1)
        .map(lambda x: x + 1)
        .collect()
    )
    print("Resultado:", resultado)

except Py4JJavaError as e:
    print("\n=== CAUSA MAIS PROFUNDA ===")
    causa = e.java_exception

    while causa.getCause() is not None:
        causa = causa.getCause()

    print(causa.toString())

    print("\n=== ERRO JAVA COMPLETO ===")
    print(e.java_exception.toString())

except Exception:
    traceback.print_exc()


Python: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
PYSPARK_PYTHON: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
Python do Spark: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
Versão do Spark: 3.5.3

=== CAUSA MAIS PROFUNDA ===
java.io.EOFException

=== ERRO JAVA COMPLETO ===
org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 1.0 failed 1 times, most recent failure: Lost task 0.0 in stage 1.0 (TID 1) (DESKTOP-8IQO9ET executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:612)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:594)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:78

In [7]:

import subprocess
import sys

comando = [
    sys.executable,
    "-c",
    "from pyspark.worker import main; print('Módulo do worker importado com sucesso')"
]

teste = subprocess.run(
    comando,
    capture_output=True,
    text=True,
    timeout=20
)

print("Código de saída:", teste.returncode)
print("Saída:", teste.stdout)
print("Erro:", teste.stderr)


Código de saída: 0
Saída: MÃ³dulo do worker importado com sucesso

Erro: 


In [8]:
%pip install --upgrade pyspark==3.5.9

     ---------------------------------------- 0.0/317.9 MB ? eta -:--:--
     ---------------------------------------- 0.3/317.9 MB ? eta -:--:--
     ---------------------------------------- 2.1/317.9 MB 8.4 MB/s eta 0:00:38
     ---------------------------------------- 2.6/317.9 MB 5.2 MB/s eta 0:01:01
     ---------------------------------------- 3.4/317.9 MB 4.9 MB/s eta 0:01:05
      --------------------------------------- 4.7/317.9 MB 5.4 MB/s eta 0:00:59
      --------------------------------------- 6.8/317.9 MB 6.0 MB/s eta 0:00:52
      --------------------------------------- 7.3/317.9 MB 6.1 MB/s eta 0:00:51
      --------------------------------------- 7.3/317.9 MB 6.1 MB/s eta 0:00:51
      --------------------------------------- 7.6/317.9 MB 4.2 MB/s eta 0:01:14
     - -------------------------------------- 8.1/317.9 MB 4.1 MB/s eta 0:01:17
     - -------------------------------------- 9.7/317.9 MB 4.3 MB/s eta 0:01:11
     - ------------------------------------- 10.7/317.

ERROR: Could not install packages due to an OSError: [WinError 32] O arquivo já está sendo usado por outro processo: 'c:\\users\\lidiane\\appdata\\local\\python\\pythoncore-3.12-64\\lib\\site-packages\\pyspark\\jars\\activation-1.1.1.jar'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pyspark

print("Versão do PySpark:", pyspark.__version__)

ImportError: cannot import name 'PySparkRuntimeError' from 'pyspark.errors' (unknown location)

In [5]:
%pip uninstall -y pyspark py4j

Found existing installation: py4j 0.10.9.7
Uninstalling py4j-0.10.9.7:
  Successfully uninstalled py4j-0.10.9.7
Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install --no-cache-dir pyspark==3.5.9 py4j==0.10.9.9

ERROR: Could not install packages due to an OSError: [WinError 32] O arquivo já está sendo usado por outro processo: 'c:\\Users\\Lidiane\\AppData\\Local\\Python\\pythoncore-3.12-64\\Lib\\site-packages\\pyspark\\jars\\HikariCP-2.5.1.jar'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/317.9 MB ? eta -:--:--
     ---------------------------------------- 0.3/317.9 MB ? eta -:--:--
     --------------------------------------- 2.4/317.9 MB 11.2 MB/s eta 0:00:29
     --------------------------------------- 2.4/317.9 MB 11.2 MB/s eta 0:00:29
     ---------------------------------------- 3.4/317.9 MB 4.8 MB/s eta 0:01:06
     ---------------------------------------- 3.7/317.9 MB 4.8 MB/s eta 0:01:05
      --------------------------------------- 4.5/317.9 MB 3.9 MB/s eta 0:01:21
      --------------------------------------- 6.6/317.9 MB 5.2 MB/s eta 0:01:01
      --------------------------------------- 6.6/317.9 MB 5.2 MB/s eta 0:01:01
      --------------------------------------- 6.6/317.9 MB 5.2 MB/s eta 0:01:01
      --------------------------------------- 7.1/317.9 MB 3.5 MB/s eta 0:01:29
      --------------------------------------- 7.3/317.9 MB 3.4 MB/s eta 0:01:31
     - -------------------------------------- 8.1/317.

ERROR: Could not install packages due to an OSError: [WinError 32] O arquivo já está sendo usado por outro processo: 'c:\\Users\\Lidiane\\AppData\\Local\\Python\\pythoncore-3.12-64\\Lib\\site-packages\\pyspark\\jars\\HikariCP-2.5.1.jar'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pyspark
import py4j

print("PySpark:", pyspark.__version__)
print("Py4J:", py4j.__version__)

PySpark: 3.5.9
Py4J: 0.10.9.9


In [4]:

import os
import sys
from pathlib import Path

ROOT = Path(
    r"E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao"
)

os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Projeto encontrado:", ROOT.is_dir())
print("Pasta common encontrada:", (ROOT / "common").is_dir())
print("Arquivo dq.py encontrado:", (ROOT / "common" / "dq.py").is_file())
print("Diretório atual:", os.getcwd())


Projeto encontrado: True
Pasta common encontrada: True
Arquivo dq.py encontrado: True
Diretório atual: E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao


In [5]:
from common.dq import DQ

print("Classe DQ importada com sucesso!")

Classe DQ importada com sucesso!


In [6]:
print(spark.version)
spark.sql("SHOW TABLES IN lake.silver").show(truncate=False)

NameError: name 'spark' is not defined

In [7]:

import os
import sys
from pathlib import Path

# Caminho do projeto
ROOT = Path(
    r"E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao"
)

# Configura o diretório e o caminho de importação
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Importa a configuração do Spark
from common import lake_local as L

# Cria a sessão Spark
spark = L.get_spark()

print("Spark iniciado com sucesso!")
print("Versão:", spark.version)
print("Modo:", spark.sparkContext.master)


Spark iniciado com sucesso!
Versão: 3.5.3
Modo: local[2]


In [8]:
print(spark.version)
spark.sql("SHOW TABLES IN lake.silver").show(truncate=False)

3.5.3
+---------+-----------------+-----------+
|namespace|tableName        |isTemporary|
+---------+-----------------+-----------+
|silver   |avaliacoes       |false      |
|silver   |produtos         |false      |
|silver   |itens_pedido_aula|false      |
|silver   |clientes         |false      |
|silver   |pedidos          |false      |
|silver   |quarentena       |false      |
|silver   |itens_pedido     |false      |
+---------+-----------------+-----------+



In [10]:
# Executa as validações diretamente via Spark SQL (sem UDFs nem workers Python)
rel = spark.sql("""
WITH c_cli AS (
    SELECT 
        count(*) AS total,
        count(CASE WHEN cliente_id IS NULL THEN 1 END) AS nulos_id,
        count(*) - count(DISTINCT cliente_id) AS dup_id,
        count(CASE WHEN email IS NULL THEN 1 END) AS nulos_email
    FROM lake.silver.clientes
),
c_ped AS (
    SELECT 
        count(*) AS total,
        count(CASE WHEN pedido_id IS NULL THEN 1 END) AS nulos_id,
        count(*) - count(DISTINCT pedido_id) AS dup_id,
        count(CASE WHEN status NOT IN ('pago', 'enviado', 'entregue', 'cancelado', 'devolvido') THEN 1 END) AS inv_status
    FROM lake.silver.pedidos
),
c_av AS (
    SELECT 
        count(*) AS total,
        count(*) - count(DISTINCT avaliacao_id) AS dup_id,
        count(CASE WHEN nota < 1 OR nota > 5 THEN 1 END) AS inv_nota
    FROM lake.silver.avaliacoes
)
SELECT 'silver.clientes' AS tabela, 'not_null' AS regra, 'cliente_id' AS coluna, total AS total_linhas, nulos_id AS linhas_invalidas, CASE WHEN nulos_id > 0 THEN 'FALHA' ELSE 'OK' END AS status FROM c_cli
UNION ALL
SELECT 'silver.clientes', 'unique', 'cliente_id', total, dup_id, CASE WHEN dup_id > 0 THEN 'FALHA' ELSE 'OK' END FROM c_cli
UNION ALL
SELECT 'silver.clientes', 'not_null', 'email', total, nulos_email, CASE WHEN nulos_email > 0 THEN 'AVISO' ELSE 'OK' END FROM c_cli
UNION ALL
SELECT 'silver.pedidos', 'not_null', 'pedido_id', total, nulos_id, CASE WHEN nulos_id > 0 THEN 'FALHA' ELSE 'OK' END FROM c_ped
UNION ALL
SELECT 'silver.pedidos', 'unique', 'pedido_id', total, dup_id, CASE WHEN dup_id > 0 THEN 'FALHA' ELSE 'OK' END FROM c_ped
UNION ALL
SELECT 'silver.pedidos', 'in_set', 'status', total, inv_status, CASE WHEN inv_status > 0 THEN 'FALHA' ELSE 'OK' END FROM c_ped
UNION ALL
SELECT 'silver.avaliacoes', 'unique', 'avaliacao_id', total, dup_id, CASE WHEN dup_id > 0 THEN 'FALHA' ELSE 'OK' END FROM c_av
UNION ALL
SELECT 'silver.avaliacoes', 'between', 'nota', total, inv_nota, CASE WHEN inv_nota > 0 THEN 'FALHA' ELSE 'OK' END FROM c_av
""")

# Grava diretamente no Iceberg
rel.writeTo("lake.silver.dq_resultados").using("iceberg").createOrReplace()

# Exibe o resultado lendo a tabela direto do catálogo
spark.table("lake.silver.dq_resultados").show(truncate=False)

+-----------------+--------+------------+------------+----------------+------+
|tabela           |regra   |coluna      |total_linhas|linhas_invalidas|status|
+-----------------+--------+------------+------------+----------------+------+
|silver.clientes  |not_null|cliente_id  |5000        |0               |OK    |
|silver.clientes  |unique  |cliente_id  |5000        |0               |OK    |
|silver.clientes  |not_null|email       |5000        |270             |AVISO |
|silver.pedidos   |not_null|pedido_id   |29960       |0               |OK    |
|silver.pedidos   |unique  |pedido_id   |29960       |0               |OK    |
|silver.pedidos   |in_set  |status      |29960       |0               |OK    |
|silver.avaliacoes|unique  |avaliacao_id|6464        |0               |OK    |
|silver.avaliacoes|between |nota        |6464        |0               |OK    |
+-----------------+--------+------------+------------+----------------+------+



In [9]:
from common.dq import DQ
cli = spark.table("lake.silver.clientes")
ped = spark.table("lake.silver.pedidos")
av  = spark.table("lake.silver.avaliacoes")

r  = DQ(spark, cli, "silver.clientes").not_null("cliente_id").unique("cliente_id").not_null("email", "aviso")
r2 = DQ(spark, ped, "silver.pedidos").not_null("pedido_id").unique("pedido_id") \
       .in_set("status", ["pago","enviado","entregue","cancelado","devolvido"])
r3 = DQ(spark, av,  "silver.avaliacoes").unique("avaliacao_id").between("nota", 1, 5)

rel = r.report()
for x in (r2, r3):
    rel = rel.unionByName(x.report())
rel.show(truncate=False)
rel.writeTo("lake.silver.dq_resultados").using("iceberg").createOrReplace()

Py4JJavaError: An error occurred while calling o193.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 39.0 failed 1 times, most recent failure: Lost task 0.0 in stage 39.0 (TID 22) (DESKTOP-8IQO9ET executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:612)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:594)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:789)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:37)
	at s

## Histórico de versões: a Silver tem memória
Cada escrita no Iceberg cria um **snapshot**: uma versão completa da tabela naquele instante. Nada é editado no lugar. Isso responde a perguntas de governança:
**o que mudou, quando, e dá para voltar?**

In [11]:
spark.sql("""
    SELECT committed_at, snapshot_id, operation,
           summary['added-records']   AS linhas_adicionadas,
           summary['total-records']   AS total_de_linhas
    FROM lake.silver.itens_pedido_aula.snapshots
    ORDER BY committed_at
""").show(truncate=False)

+-----------------------+------------------+---------+------------------+---------------+
|committed_at           |snapshot_id       |operation|linhas_adicionadas|total_de_linhas|
+-----------------------+------------------+---------+------------------+---------------+
|2026-10-24 13:29:29.075|728372584219681079|append   |50105             |50105          |
+-----------------------+------------------+---------+------------------+---------------+



Vamos simular um erro humano: alguém "limpa" os pedidos cancelados achando que eram lixo.
Guardamos o id da versão **antes** do estrago.

In [12]:
TABELA = "lake.silver.itens_pedido_aula"

snap_bom = spark.sql(f"SELECT snapshot_id FROM {TABELA}.snapshots ORDER BY committed_at DESC LIMIT 1").first()[0]
antes = spark.table(TABELA).count()

spark.sql(f"DELETE FROM {TABELA} WHERE status = 'cancelado'")        # a regra errada

depois = spark.table(TABELA).count()
print(f"linhas: {antes} -> {depois}  (sumiram {antes - depois})")
print("snapshot bom:", snap_bom)

linhas: 50105 -> 43886  (sumiram 6219)
snapshot bom: 728372584219681079


In [13]:
# voltar no tempo (só ler)
spark.sql(f"SELECT count(*) AS linhas_na_versao_antiga FROM {TABELA} VERSION AS OF {snap_bom}").show()
spark.sql(f"SELECT count(*) AS linhas_na_versao_atual   FROM {TABELA}").show()

+-----------------------+
|linhas_na_versao_antiga|
+-----------------------+
|                  50105|
+-----------------------+

+----------------------+
|linhas_na_versao_atual|
+----------------------+
|                 43886|
+----------------------+



In [14]:

spark.sql(f"CALL lake.system.rollback_to_snapshot('silver.itens_pedido_aula', {snap_bom})")
print("linhas depois do rollback:", spark.table(TABELA).count())

linhas depois do rollback: 50105


Se alguma regra aparece com status `AVISO` ou `ERRO`, ótimo: é o ponto de conversa. (Dica: e-mail inválido vira `NULL` na Silver, e a regra `not_null(email)` aparece como `AVISO`: o cliente continua existindo, só perdeu o contato.)

## Checkpoint
- [ ] Silver com 5 tabelas + `quarentena` + `dq_resultados`.
- [ ] Você consegue explicar por que um pedido foi para a quarentena.

➡️ Próximo: `03_gold.ipynb` (o foco da aula)